In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

# XGBoost
We'll use in this notebook an external library

## Load Data & Preparation
Let's load the clean Airbnb dataset in again , but this time we will split in 3, so we have a validation set as well as a training and test sets
We created it in a previous notebook, it should exists in `/home/jovyan/work/datasets/outpus/airbnb/clean_data` 
Also, let's index all of our categorical features, and set our label to be **`log(price)`**.

In [ ]:
from pyspark.sql.functions import log, col
from pyspark.ml.feature import StringIndexer, VectorAssembler

file_path = "/home/jovyan/work/datasets/outpus/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)
train_df, test_df = airbnb_df.withColumn(<TODO>, <TODO>).randomSplit([.8, .2], seed=42)

#Select all categorical columns and index them 
categorical_cols = <TODO>
index_output_cols = <TODO>

string_indexer = StringIndexer(inputCols=<TODO>, outputCols=<TODO>, handleInvalid="skip")

#Select all numeric columns except price and label
numeric_cols = <TODO>

assembler_inputs = <TODO>
vec_assembler = VectorAssembler(inputCols=<TODO>, outputCol=<TODO>)

### Distributed Training of XGBoost Models
We create an SparkXGBRegressor with the following params as a json
* n_estimators: 100
* learning_rate: 0.1
* max_depth: 4
* random_state:42
* missing:0

In [ ]:
from xgboost.spark import SparkXGBRegressor
from pyspark.ml import Pipeline

params = {"n_estimators": 100, "learning_rate": 0.1, "max_depth": 4, "random_state": 42, "missing": 0}

xgboost = SparkXGBRegressor(**params)

pipeline = Pipeline(stages=[<TODO>])
pipeline_model = pipeline.fit(<TODO>)

## Evaluate Model Performance
* Remember to exponentiate the label column so we can evaluate the real prediction
* Set the exp(prediction) in a new column called just prediction

In [ ]:
from pyspark.sql.functions import exp, col

log_pred_df = pipeline_model.transform(<TODO>)

exp_xgboost_df = log_pred_df.withColumn(<TODO>, <TODO>)

display(exp_xgboost_df.select(<TODO>, <TODO>))

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

regression_evaluator = <TODO>

rmse = regression_evaluator.evaluate(<TODO>)
r2 = regression_evaluator.setMetricName(<TODO>).evaluate(<TODO>)
print(f"RMSE is {rmse}")
print(f"R2 is {r2}")